# Reciprocal-Space Mapping

The public RSM path separates a source, persisted geometry, and an
`RSMPlan`. Smoke mode creates a bounded `RSMVolume` for slice review;
real mode requires the matching `PixelQMap` and scan motor mapping.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from xrd_tools.analysis import RSMPlan, run_rsm
from xrd_tools.io import open_scan
from xrd_tools.rsm import RSMVolume
from xrd_tools.viz import plot_image


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
processed_file = TEST_DATA / "processed.nxs"
mapper = None  # Real mode: PixelQMap from the experiment's persisted geometry.
diff_motors = ()  # Real mode: one persisted scan_data motor name per circle.
slice_axis = widgets.Dropdown(options=("h", "k", "l"), value="l", description="integrate")
compute = widgets.Button(description="Compute RSM", button_style="primary")
status = widgets.HTML("<i>Compute is explicit; changing a slice does not regrid data.</i>")
display(widgets.VBox([slice_axis, compute, status]))


In [ ]:
if SMOKE_MODE:
    h = np.linspace(-0.08, 0.08, 24)
    k = np.linspace(-0.06, 0.06, 20)
    l = np.linspace(0.90, 1.10, 18)
    hh, kk, ll = np.meshgrid(h, k, l, indexing="ij")
    volume = RSMVolume(h, k, l, np.exp(-0.5 * ((hh / 0.02) ** 2 + (kk / 0.018) ** 2 + ((ll - 1.0) / 0.03) ** 2)))
else:
    assert processed_file.is_file(), f"Missing processed NeXus: {processed_file}"
    assert mapper is not None and diff_motors, "Set mapper and diff_motors from the experiment geometry."
    source = open_scan(processed_file)
    plan = RSMPlan(mapper=mapper, diff_motors=tuple(diff_motors), bins=(96, 96, 96))
    volume = run_rsm(plan, source).payload

axis_a, axis_b, image, integrated = volume.get_slice(slice_axis.value)
fig, ax = plt.subplots(figsize=(6, 4))
plot_image(ax, image.T, attrs={"xlabel": "axis 1", "ylabel": "axis 2", "title": f"RSM projection over {slice_axis.value}"}, cb_label="Intensity")
plt.show()
{"shape": volume.shape, "integrated_points": len(integrated), "bounds": volume.get_bounds()}
